In [1]:
# Instalar dependencias necesarias de forma segura
import sys
import subprocess

packages = [
    'pandas',
    'numpy',
    'matplotlib',
    'seaborn',
    'scikit-learn',
    'imbalanced-learn',
    'openpyxl'
]

for pkg in packages:
    try:
        if pkg == 'scikit-learn':
            __import__('sklearn')
        elif pkg == 'imbalanced-learn':
            __import__('imblearn')
        else:
            __import__(pkg)
        print(f"✓ {pkg} ya está instalado")
    except ImportError:
        print(f'Instalando {pkg}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
        print(f"✓ {pkg} instalado correctamente")

# Importar librerías
try:
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
    import seaborn as sns
    from sklearn.preprocessing import StandardScaler, LabelEncoder
    from sklearn.model_selection import train_test_split
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.pipeline import Pipeline
    from sklearn.metrics import classification_report, f1_score, confusion_matrix
    from imblearn.over_sampling import SMOTE
    print("\n✓ Todas las librerías importadas correctamente")
except Exception as e:
    print('Error en importaciones:', e)
    raise

✓ pandas ya está instalado
✓ numpy ya está instalado
✓ matplotlib ya está instalado
✓ seaborn ya está instalado
✓ scikit-learn ya está instalado
Instalando imbalanced-learn...
✓ imbalanced-learn instalado correctamente
✓ openpyxl ya está instalado

✓ Todas las librerías importadas correctamente


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, LabelEncoder

# 1. Load the dataset
df = pd.read_excel("Dry_Bean_Dataset.xlsx")

# Visualize if there are any initial null values
print("Missing data analysis by column:")
print(df.isnull().sum())

# 2. Separate features (X) and the target variable (y)
X = df.drop(columns=['Class'])
y = df['Class']

# 3. Categorical Variable Encoding (Target)
# We use LabelEncoder because 'Class' is the multi-classification target variable
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# 4. Split the dataset into Train and Test sets (CRITICAL to avoid Data Leakage)
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)

# 5. Missing Data Imputation Strategy
# Although the Dry Beans dataset doesn't usually have nulls, it's good practice to include an imputer.
# We use the median because it is less sensitive to outliers in geometric features.
imputer = SimpleImputer(strategy='median')

# IMPORTANT: We apply fit_transform ONLY on the training set
X_train_imputed = imputer.fit_transform(X_train)
# For the test set, we only apply transform (using the medians calculated from the train set)
X_test_imputed = imputer.transform(X_test)

# 6. Standardization / Normalization
# We use StandardScaler so that variables have a mean of 0 and a standard deviation of 1.
# This is vital because 'Area' is in the tens of thousands and the 'ShapeFactors' are very small decimals.
scaler = StandardScaler()

# Again, fit_transform on Train, and only transform on Test
X_train_scaled = scaler.fit_transform(X_train_imputed)
X_test_scaled = scaler.transform(X_test_imputed)

# Convert back to DataFrames (optional, for better visualization or subsequent handling)
feature_names = X.columns
X_train_final = pd.DataFrame(X_train_scaled, columns=feature_names)
X_test_final = pd.DataFrame(X_test_scaled, columns=feature_names)

print("\nPreprocessing completed successfully.")
print(f"X_train_final dimensions: {X_train_final.shape}")
print(f"X_test_final dimensions: {X_test_final.shape}")

Missing data analysis by column:
Area               0
Perimeter          0
MajorAxisLength    0
MinorAxisLength    0
AspectRation       0
Eccentricity       0
ConvexArea         0
EquivDiameter      0
Extent             0
Solidity           0
roundness          0
Compactness        0
ShapeFactor1       0
ShapeFactor2       0
ShapeFactor3       0
ShapeFactor4       0
Class              0
dtype: int64

Preprocessing completed successfully.
X_train_final dimensions: (10888, 16)
X_test_final dimensions: (2723, 16)


In [7]:
# Import the balancing libraries
from imblearn.over_sampling import SMOTE, RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler
from collections import Counter

# Visualize the original count in the Train Set
print(f"Original distribution in y_train: {Counter(y_train)}")

# ---------------------------------------------------------
# TECHNIQUE 1: Random Undersampling
# ---------------------------------------------------------
# Reduces all classes to the size of the minority class (BOMBAY: 418)
rus = RandomUnderSampler(random_state=42)
X_train_rus, y_train_rus = rus.fit_resample(X_train_final, y_train)
print(f"Distribution after UnderSampling: {Counter(y_train_rus)}")

# ---------------------------------------------------------
# TECHNIQUE 2: Random Oversampling
# ---------------------------------------------------------
# Duplicates records from minority classes until they match the majority class (DERMASON: 2837)
ros = RandomOverSampler(random_state=42)
X_train_ros, y_train_ros = ros.fit_resample(X_train_final, y_train)
print(f"Distribution after OverSampling: {Counter(y_train_ros)}")

# ---------------------------------------------------------
# TECHNIQUE 3: SMOTE (Synthetic Minority Over-sampling Technique)
# ---------------------------------------------------------
# Creates SYNTHETIC samples (not duplicates) for the minority classes
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_final, y_train)
print(f"Distribution after SMOTE: {Counter(y_train_smote)}")

# Note: The test set (X_test_final, y_test) is kept INTACT to evaluate the model with real data.

Original distribution in y_train: Counter({np.int64(3): 2837, np.int64(6): 2109, np.int64(5): 1621, np.int64(4): 1542, np.int64(2): 1304, np.int64(0): 1057, np.int64(1): 418})
Distribution after UnderSampling: Counter({np.int64(0): 418, np.int64(1): 418, np.int64(2): 418, np.int64(3): 418, np.int64(4): 418, np.int64(5): 418, np.int64(6): 418})
Distribution after OverSampling: Counter({np.int64(5): 2837, np.int64(4): 2837, np.int64(2): 2837, np.int64(6): 2837, np.int64(3): 2837, np.int64(0): 2837, np.int64(1): 2837})
Distribution after SMOTE: Counter({np.int64(5): 2837, np.int64(4): 2837, np.int64(2): 2837, np.int64(6): 2837, np.int64(3): 2837, np.int64(0): 2837, np.int64(1): 2837})


In [9]:
# Join X and y to conveniently save them in CSVs
# 1. Balanced Training Set (SMOTE) - THE MAIN ONE
train_smote_df = pd.DataFrame(X_train_smote, columns=feature_names)
train_smote_df['Class'] = y_train_smote
train_smote_df.to_csv("train_data_SMOTE.csv", index=False)

# 2. Original Training Set (Imbalanced) - FOR COMPARISON IN STEP 5
train_original_df = X_train_final.copy()
train_original_df['Class'] = y_train
train_original_df.to_csv("train_data_IMBALANCED.csv", index=False)

# 3. Test Set - INTACT, ONLY FOR FINAL EVALUATION
test_df = X_test_final.copy()
test_df['Class'] = y_test
test_df.to_csv("test_data_FINAL.csv", index=False)

print("Files successfully exported.")

Files successfully exported.


In [ ]:
# =====================================================================
# APARTADO 3: ANÁLISIS EXPLORATORIO ADICIONAL
# =====================================================================
print("\n--- 3. ANÁLISIS EXPLORATORIO DE DATOS ---")

# Visualización de desbalanceo en datos resampled
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
sns.countplot(x=y, palette='Set2')
plt.title('Dataset Original: Desbalanceo de Clases')
plt.xlabel('Clase')
plt.ylabel('Cantidad')
plt.xticks(rotation=45)

plt.subplot(1, 2, 2)
sns.countplot(x=y_resampled_labels, palette='Set2')
plt.title('Después de SMOTE: Clases Balanceadas')
plt.xlabel('Clase')
plt.ylabel('Cantidad')
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

# Histogramas de distribuciones de características
print("\nDistribuciones de características (primeras 4 características):")
X_scaled_df.iloc[:, :4].hist(figsize=(12, 5), bins=30)
plt.tight_layout()
plt.show()

# Matriz de correlaciones
print("\nMatriz de correlación de características:")
plt.figure(figsize=(10, 8))
sns.heatmap(X.corr(), annot=False, cmap='coolwarm', square=True)
plt.title('Correlación entre características')
plt.tight_layout()
plt.show()

print("\n✓ Análisis exploratorio completado")

In [ ]:
# =====================================================================
# APARTADO 4: DIVISION EN CONJUNTOS DE ENTRENAMIENTO Y PRUEBA
# =====================================================================
print("\n--- 4. DIVISIÓN TRAIN/TEST ---")

# División estratificada para mantener proporción de clases
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Tamaño del conjunto de entrenamiento: {X_train.shape}")
print(f"Tamaño del conjunto de prueba: {X_test.shape}")

# Estandarización de características
scaler_final = StandardScaler()
X_train_scaled = scaler_final.fit_transform(X_train)
X_test_scaled = scaler_final.transform(X_test)

print("\n✓ Escalado completado")
print(f"  - Media de características en train (primera característica): {X_train_scaled[:, 0].mean():.4f}")
print(f"  - Desv. Estándar en train (primera característica): {X_train_scaled[:, 0].std():.4f}")

array([[-0.3713618 , -0.53113068, -0.69536968, ...,  1.11808237,
         1.48524952,  0.95141553],
       [ 0.02895235,  0.49881788,  0.78351767, ..., -1.32149747,
        -1.87953412,  0.15816127],
       [ 0.72707783,  0.79500482,  0.82438155, ..., -0.7876742 ,
        -0.2375073 , -0.35302543],
       ...,
       [ 0.43664179,  0.87746349,  0.47987974, ..., -0.53535794,
        -0.01143779, -1.4807801 ],
       [ 0.01662606,  0.31205264,  0.64565274, ..., -1.18140133,
        -1.61703817, -0.74823181],
       [ 4.01901639,  3.47586447,  3.51587226, ..., -1.68315117,
        -0.80825296, -0.89219251]], shape=(10888, 16))

In [ ]:
# =====================================================================
# APARTADO 5: MODELADO - MÉTODO A: RandomForest + class_weight
# =====================================================================
print("\n--- 5. MODELO A: RandomForest con class_weight='balanced' ---")

# Crear y entrenar RandomForest
rf_a = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

print("Entrenando RandomForest (Método A)...")
rf_a.fit(X_train_scaled, y_train)
y_pred_a = rf_a.predict(X_test_scaled)

print("\n=== RESULTADOS: RandomForest + class_weight='balanced' ===")
print(classification_report(y_test, y_pred_a))
print("\nMacro F1 Score:", f1_score(y_test, y_pred_a, average="macro"))

print("\nMatriz de Confusión:")
cm_a = confusion_matrix(y_test, y_pred_a)
print(cm_a)

print("\n✓ Modelo A completado")

C:\Users\alex.iturgaitz\AppData\Roaming\Python\Python312\site-packages\sklearn\base.py:1336: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


=== RandomForest + class_weight='balanced' ===
              precision    recall  f1-score   support

    BARBUNYA       0.94      0.89      0.91       265
      BOMBAY       1.00      1.00      1.00       104
        CALI       0.94      0.94      0.94       326
    DERMASON       0.91      0.92      0.92       709
       HOROZ       0.97      0.95      0.96       386
       SEKER       0.94      0.96      0.95       406
        SIRA       0.86      0.85      0.85       527

    accuracy                           0.92      2723
   macro avg       0.93      0.93      0.93      2723
weighted avg       0.92      0.92      0.92      2723

Macro F1: 0.9326187597475176

Matriz de confusión:
[[236   0  17   0   1   2   9]
 [  0 104   0   0   0   0   0]
 [ 10   0 305   0   5   2   4]
 [  0   0   0 655   0  10  44]
 [  1   0   4   4 368   0   9]
 [  1   0   0   6   0 389  10]
 [  4   0   0  55   7  11 450]]


In [ ]:
# =====================================================================
# APARTADO 6: MODELADO - MÉTODO B: SMOTE + RandomForest en Pipeline
# =====================================================================
print("\n--- 6. MODELO B: Pipeline con SMOTE + RandomForest ---")

# Crear pipeline con SMOTE + RandomForest
smote_pipe = SMOTE(random_state=42)

rf_b = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

pipe_b = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("smote", smote_pipe),
    ("rf", rf_b),
])

print("Entrenando Pipeline (Método B)...")
pipe_b.fit(X_train, y_train)
y_pred_b = pipe_b.predict(X_test)

print("\n=== RESULTADOS: SMOTE + RandomForest en Pipeline ===")
print(classification_report(y_test, y_pred_b))
print("\nMacro F1 Score:", f1_score(y_test, y_pred_b, average="macro"))

print("\nMatriz de Confusión:")
cm_b = confusion_matrix(y_test, y_pred_b)
print(cm_b)

print("\n✓ Modelo B completado")

# =====================================================================
# COMPARACIÓN DE MÉTODOS
# =====================================================================
print("\n--- COMPARACIÓN DE MÉTODOS ---")
f1_a = f1_score(y_test, y_pred_a, average="macro")
f1_b = f1_score(y_test, y_pred_b, average="macro")

print(f"\nMétodo A (class_weight):  F1 Macro = {f1_a:.4f}")
print(f"Método B (SMOTE):         F1 Macro = {f1_b:.4f}")
print(f"Mejor método: {'A (class_weight)' if f1_a > f1_b else 'B (SMOTE)' if f1_b > f1_a else 'Empate'}")

C:\Users\alex.iturgaitz\AppData\Roaming\Python\Python312\site-packages\sklearn\base.py:1336: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


=== SMOTE + RandomForest ===
              precision    recall  f1-score   support

    BARBUNYA       0.92      0.90      0.91       265
      BOMBAY       1.00      1.00      1.00       104
        CALI       0.94      0.94      0.94       326
    DERMASON       0.92      0.91      0.91       709
       HOROZ       0.96      0.95      0.96       386
       SEKER       0.94      0.96      0.95       406
        SIRA       0.86      0.86      0.86       527

    accuracy                           0.92      2723
   macro avg       0.93      0.93      0.93      2723
weighted avg       0.92      0.92      0.92      2723

Macro F1: 0.9325596766874037

Matriz de confusión:
[[239   0  15   0   1   3   7]
 [  0 104   0   0   0   0   0]
 [ 12   0 305   0   5   2   2]
 [  0   0   0 647   1  12  49]
 [  2   0   5   4 368   0   7]
 [  1   0   0   5   0 390  10]
 [  6   0   0  50   7  10 454]]
